# Project 1 — Your local technical tutor

- **Input:** a technical question and your learning preferences.
- **Output:** a streamed explanation, rendered Markdown, and a saved answer.
- **Inference:** local Ollama only; no API key.
- This self-contained notebook completes the Week 1 tutor exercise. It does not import files from the earlier package.

## 1. Local setup

- Install [Ollama](https://ollama.com/download) if needed and open the Ollama application.
- In a terminal, run `ollama pull llama3.2:3b` once (approximately 2 GB).
- For a smaller model, use `ollama pull llama3.2:1b` and change `MODEL` below.
- Open this notebook in Jupyter, VS Code, or Cursor with a Python 3.10+ kernel.
- Run the cells from top to bottom. No OpenAI key or model SDK is required.
- Model calls go to the loopback address below. Downloading the model needs internet; inference uses your local machine.
- If the server is not running, start the Ollama app or run `ollama serve` in a separate terminal.

In [ ]:
import json
from pathlib import Path
from urllib.request import Request, build_opener, ProxyHandler
from urllib.error import HTTPError, URLError
from urllib.parse import urlsplit
# Jupyter renders Markdown when IPython is present; the fallback keeps the
# code cells runnable in a plain Python 3.10+ interpreter too.
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(value):
        print(value)

MODEL = "llama3.2:3b"  # Or "llama3.2:1b" after pulling that model.
OLLAMA_URL = "http://127.0.0.1:11434"
NUM_CTX = 8192
NUM_PREDICT = 900
TIMEOUT_SECONDS = 300  # CPU-only inference may be slow.
RESULTS_DIR = Path.cwd() / "results"

In [ ]:
# A small direct client for Ollama's native API; no extra model SDK needed.
def local_request(path, payload=None, timeout=TIMEOUT_SECONDS):
    if not path.startswith("/") or path.startswith("//"):
        raise ValueError("Ollama API paths must start with a single slash.")
    parsed = urlsplit(OLLAMA_URL)
    try:
        parsed.port  # Surface malformed-port errors before making a request.
    except ValueError as exc:
        raise ValueError("OLLAMA_URL must contain a valid port.") from exc
    if (parsed.scheme != "http" or parsed.hostname not in {"127.0.0.1", "localhost", "::1"}
            or parsed.username or parsed.password or parsed.path not in {"", "/"}
            or parsed.query or parsed.fragment):
        raise ValueError("Use an HTTP loopback Ollama URL, e.g. http://127.0.0.1:11434")
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = Request(OLLAMA_URL.rstrip("/") + path, data=data,
                  headers={"Content-Type": "application/json"})
    # Bypass system HTTP proxies for the local model endpoint.
    return build_opener(ProxyHandler({})).open(req, timeout=timeout)

def check_ollama():
    try:
        with local_request("/api/tags", timeout=5) as response:
            document = json.load(response)
        if not isinstance(document, dict) or not isinstance(document.get("models"), list):
            raise ValueError("Ollama returned an unexpected /api/tags response.")
        names = [m["name"] for m in document["models"]
                 if isinstance(m, dict) and isinstance(m.get("name"), str)]
    except (OSError, ValueError) as exc:
        raise RuntimeError("Cannot reach local Ollama. Start the Ollama app or run ollama serve, then rerun this cell.") from exc
    target = MODEL.strip()
    if not target:
        raise RuntimeError("MODEL must be a non-empty Ollama model name.")
    targets = {target} if ":" in target.rsplit("/", 1)[-1] else {target, target + ":latest"}
    if not targets.intersection(names):
        raise RuntimeError(f"Model {MODEL!r} is not installed. Run: ollama pull {MODEL}. Installed models: {names}")
    print(f"Ready: {MODEL} at {OLLAMA_URL}")
    return names

def chat(messages, *, json_mode=False, show_stream=True, max_tokens=NUM_PREDICT):
    payload = {"model": MODEL, "messages": messages, "stream": True,
               "options": {"temperature": 0.2, "num_ctx": NUM_CTX, "num_predict": max_tokens},
               "keep_alive": "5m"}
    if json_mode:
        payload["format"] = "json"
    parts, finished = [], False
    try:
        with local_request("/api/chat", payload) as response:
            for line in response:
                if not line.strip():
                    continue
                event = json.loads(line)
                if event.get("error"):
                    raise RuntimeError("Ollama returned an error: " + str(event["error"]))
                text = event.get("message", {}).get("content", "")
                if text:
                    parts.append(text)
                    if show_stream:
                        print(text, end="", flush=True)
                if event.get("done"):
                    if event.get("done_reason") == "length":
                        raise RuntimeError("Output reached the token limit. Increase NUM_PREDICT/max_tokens and rerun.")
                    finished = True
                    break
    except HTTPError as exc:
        raise RuntimeError(f"Ollama HTTP {exc.code}. Check that MODEL is installed and supports chat.") from exc
    except (OSError, ValueError) as exc:
        raise RuntimeError("Local model request failed. Check Ollama and the timeout; rerun this cell.") from exc
    finally:
        if show_stream:
            print()
    result = "".join(parts).strip()
    if not finished or not result:
        raise RuntimeError("Ollama returned an empty or incomplete response; no result was saved.")
    return result

In [ ]:
check_ollama()

## 2. Customize your question and tutor

In [ ]:
QUESTION = '''Please explain what this Python code does and why:
yield from {book.get("author") for book in books if book.get("author")}'''
LEARNER_BACKGROUND = "I know basic Python and want clear, practical explanations."
PREFERRED_STYLE = "Use bullet points, one small runnable example, and common mistakes."

# Add good input/answer pairs here to experiment with few-shot prompting.
EXAMPLES = []  # [{"question": "...", "answer": "..."}]

def tutor_messages(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty technical question.")
    messages = [{"role": "system", "content": (
        "You are a careful technical tutor. Explain accurately, state assumptions, "
        "and distinguish what code does from why someone would use it. "
        "Do not execute user code. If unsure, say so. "
        f"Learner: {LEARNER_BACKGROUND}. Style: {PREFERRED_STYLE}")}]
    for example in EXAMPLES:
        messages.extend([{"role": "user", "content": example["question"]},
                         {"role": "assistant", "content": example["answer"]}])
    messages.append({"role": "user", "content": question.strip()})
    return messages

## 3. Ask your local model
Run this cell again after changing the question. Each run starts a fresh conversation.

In [ ]:
answer = chat(tutor_messages(QUESTION))
display(Markdown(answer))

## 4. Save your answer

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
answer_path = RESULTS_DIR / "technical-answer.md"
answer_path.write_text(f"# Technical tutor\n\nModel: {MODEL}\n\n## Question\n{QUESTION}\n\n## Answer\n{answer}\n", encoding="utf-8")
print("Saved:", answer_path.resolve())

## 5. Check the answer for the starter question

- The comprehension filters out falsey author values and removes duplicates.
- A set does not guarantee useful ordering; author values must be hashable.
- The set is built eagerly before the first author is yielded.
- `yield from` yields each element of the set from the surrounding generator function.
- Check that the response distinguishes a set comprehension from a lazy generator expression.
- Replace the starter question and rerun the input and answer cells for your next topic.

## Sources and experiments

- [Instructor Week 1 materials](https://github.com/ed-donner/llm_engineering/tree/main/week1)
- [Final assignment lecture](https://vodafoneegypt.udemy.com/course/llm-engineering-master-ai-and-large-language-models/learn/lecture/52939993)
- [Ollama chat API](https://docs.ollama.com/api/chat)
- [Ollama model list API](https://docs.ollama.com/api/tags)
- [Llama 3.2 model](https://ollama.com/library/llama3.2)

Change one input, model, or prompt at a time. Compare correctness, clarity, latency, and unsupported claims. Generated output needs review; a small local model can make mistakes.